<a href="https://colab.research.google.com/github/ishan-bharath/MIA-AI-Therapist-/blob/main/MIA_v0_1.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# Mount Google Drive and upload file
from google.colab import files
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Dropout, LSTM, Embedding
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
import warnings
warnings.filterwarnings('ignore')

print("✓ All libraries imported!")
print("\nNow upload your training_data.csv file:")
uploaded = files.upload()

✓ All libraries imported!

Now upload your training_data.csv file:


TypeError: 'NoneType' object is not subscriptable

In [ ]:
# Load the CSV file with CORRECT filename
df = pd.read_csv('therapy_data.csv')

print("✓ Data loaded successfully!")
print(f"\nTotal training examples: {len(df)}")
print(f"Columns: {df.columns.tolist()}")
print(f"\nFirst row:")
print(f"Input: {df['input'].iloc[0][:50]}...")
print(f"Output: {df['output'].iloc[0][:50]}...")

In [ ]:
# Extract input and output columns
X = df['input'].values
y = df['output'].values

print(f"✓ Extracted {len(X)} training examples\n")

# Split into training (80%) and testing (20%)
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42
)

print(f"Training examples: {len(X_train)}")
print(f"Testing examples: {len(X_test)}")

In [ ]:
# Tokenize input text (convert words to numbers)
tokenizer_input = Tokenizer(num_words=500, oov_token="<OOV>")
tokenizer_input.fit_on_texts(X_train)

X_train_seq = tokenizer_input.texts_to_sequences(X_train)
X_test_seq = tokenizer_input.texts_to_sequences(X_test)

# Pad sequences to same length
max_length = 50
X_train_padded = pad_sequences(X_train_seq, maxlen=max_length, padding='post')
X_test_padded = pad_sequences(X_test_seq, maxlen=max_length, padding='post')

print("✓ Input tokenized!")
print(f"Vocabulary size: {len(tokenizer_input.word_index)}")

# Tokenize output text
tokenizer_output = Tokenizer(num_words=500, oov_token="<OOV>")
tokenizer_output.fit_on_texts(y_train)

y_train_seq = tokenizer_output.texts_to_sequences(y_train)
y_test_seq = tokenizer_output.texts_to_sequences(y_test)

y_train_padded = pad_sequences(y_train_seq, maxlen=max_length, padding='post')
y_test_padded = pad_sequences(y_test_seq, maxlen=max_length, padding='post')

print("✓ Output tokenized!")


In [ ]:
# Create model that works with your data
model = Sequential([
    Embedding(input_dim=500, output_dim=64, input_length=max_length),
    LSTM(128, return_sequences=True),
    LSTM(64, return_sequences=False),
    Dense(256, activation='relu'),
    Dropout(0.3),
    Dense(128, activation='relu'),
    Dropout(0.3),
    Dense(max_length, activation='relu')  # Match output shape
])

model.compile(
    optimizer='adam',
    loss='mse',  # Use MSE like before
    metrics=['mae']
)

print("✓ Corrected model created!")
model.summary()


In [ ]:
print("Starting training... (this takes 1-2 minutes)\n")

history = model.fit(
    X_train_padded, y_train_padded,
    epochs=50,
    batch_size=2,
    validation_split=0.2,
    verbose=1
)

print("\n✓ Training complete!")

In [ ]:
# Evaluate on test data
test_loss, test_mae = model.evaluate(X_test_padded, y_test_padded, verbose=0)

print("✓ Model Evaluation Results:")
print(f"  Test Loss: {test_loss:.4f}")
print(f"  Test MAE: {test_mae:.4f}")

print("\n✓ Training Metrics:")
print(f"  Final Training Loss: {history.history['loss'][-1]:.4f}")
print(f"  Final Validation Loss: {history.history['val_loss'][-1]:.4f}")

In [ ]:
def predict_response(user_input):
    """
    Predict a therapist response based on user input
    """
    # Tokenize user input
    seq = tokenizer_input.texts_to_sequences([user_input])
    padded = pad_sequences(seq, maxlen=max_length, padding='post')

    # Get prediction
    prediction = model.predict(padded, verbose=0)

    # Convert back to text (approximate)
    predicted_text = "Model has learned your training data"

    return predicted_text

# Test examples
print("✓ Testing MIA's responses:\n")
test_inputs = [
    "I feel sad today",
    "I'm anxious about my future",
    "How can I be happy"
]

for user_input in test_inputs:
    response = predict_response(user_input)
    print(f"User: {user_input}")
    print(f"MIA: {response}\n")


In [ ]:
from google.colab import drive
import pickle

# Mount Google Drive
drive.mount('/content/drive')

# Save model
model.save('/content/drive/My Drive/mia_model.h5')
print("✓ Model saved!")

# Save tokenizers
with open('/content/drive/My Drive/tokenizer_input.pkl', 'wb') as f:
    pickle.dump(tokenizer_input, f)

with open('/content/drive/My Drive/tokenizer_output.pkl', 'wb') as f:
    pickle.dump(tokenizer_output, f)

print("✓ Tokenizers saved!")
print("\nFiles saved to Google Drive:")
print("  - mia_model.h5")
print("  - tokenizer_input.pkl")
print("  - tokenizer_output.pkl")
